In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import pyodbc
import numpy as np
import pandas as pd
from config import server, database, username, password
from datetime import datetime
import os
import shutil
import win32com.client as win32
import win32com.client.gencache


connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)

conn = pyodbc.connect(connection_string)

with open('my_query1.sql', 'r') as file:
    query = file.read()

df = pd.read_sql(query, conn)

df_transitos = pd.read_csv('transitos.csv', encoding='latin1')
df_transitos['Fecha entrega'] = pd.to_datetime(df_transitos['Fecha entrega'], format='%d/%m/%Y', errors='coerce')

hoy = pd.to_datetime(datetime.today().date())
df_transitos = df_transitos[df_transitos['Fecha entrega'] >= hoy]
df_transitos['Cantidad'] = pd.to_numeric(df_transitos['Cantidad'], errors='coerce')

transit_sums = df_transitos.groupby(['Solicitante', 'Material'])['Cantidad'].sum().reset_index()

df = df.merge(transit_sums, how='left', left_on=['Sap', 'Sku'], right_on=['Solicitante', 'Material'])
df.rename(columns={'Cantidad': 'Piezas_transito'}, inplace=True)
df.drop(columns=['Solicitante', 'Material'], inplace=True)

df['OH_proyectado'] = round(df['OH_Piezas'].fillna(0) + df['Piezas_transito'].fillna(0), 0)

df['Piezas_empuje'] = np.where(
    (df['Tipo_merma'] == 'Ok') & (df['Inventario_sugerido'] > df['OH_proyectado']),
    round(df['Inventario_sugerido'] - df['OH_proyectado'], 0),
    0
)

df['No_cargar_pedido'] = np.where(
    ((df['Tipo_merma'] != 'Ok') & (df['Tipo_merma'] != 'Alta')) & (df['OH_proyectado'] > df['Inventario_sugerido']),
    1,
    0
)

condiciones = [
    (df['Piezas_empuje'] > 0) & (df['Top_venta'] == 1),
    (df['Piezas_empuje'] > 0) & (df['Top_venta'] == 0)
]


valores = [1, 2]

df['Prioridad'] = np.select(condiciones, valores, default=0)


df['Empuje_cabecera'] = np.where(
    (df['Objetivo_cabecera'] - df['OH_proyectado'] > 0) & (df['Con_cabecera_activa'] == 1),
    df['Objetivo_cabecera'] - df['OH_proyectado'],
    0
)

condicion = (df['Piezas_empuje'] > 0) & (df['linea'] == 'Q-FRESCOS')
df.loc[condicion, 'Piezas_empuje'] = np.round(df.loc[condicion, 'Piezas_empuje'] * 0.33, 0)

df.to_csv('base.csv', index=False)


folder_name = 'emergencia'
folder_path = os.path.join(os.getcwd(), folder_name)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Carpeta '{folder_name}' creada.")
else:
    print(f"Carpeta '{folder_name}' ya existe.")
for root, dirs, files in os.walk(folder_path, topdown=False):
    for name in files:
        file_path = os.path.join(root, name)
        try:
            os.remove(file_path)
            print(f"Archivo eliminado: {file_path}")
        except Exception as e:
            print(f"No se pudo eliminar el archivo {file_path}. Error: {e}")
    for name in dirs:
        dir_path = os.path.join(root, name)
        try:
            os.rmdir(dir_path)
            print(f"Subcarpeta eliminada: {dir_path}")
        except Exception as e:
            print(f"No se pudo eliminar la subcarpeta {dir_path}. Error: {e}")

            
excel_file = 'Ajustes_pedido.xlsx'
folder_name = 'emergencia'
folder_path = os.path.join(os.getcwd(), folder_name)
source_path = os.path.join(os.getcwd(), excel_file)
destination_path = os.path.join(folder_path, excel_file)
try:
    shutil.copy2(source_path, destination_path)
    print(f"Archivo '{excel_file}' copiado exitosamente a la carpeta '{folder_name}'.")
except Exception as e:
    print(f"No se pudo copiar el archivo. Error: {e}")


connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)

conn = pyodbc.connect(connection_string)

with open('my_query5.sql', 'r') as file:
    query_rol = file.read()

df_rol = pd.read_sql(query_rol, conn)

csv_path = os.path.join(os.getcwd(), 'emergencia', 'df_rol.csv')

df_rol.to_csv(csv_path, index=False)
print(f"Archivo guardado en: {csv_path}")

win32com.client.gencache.is_readonly = True

excel_file = os.path.abspath("Ajustes_pedido.xlsx")

excel = win32.Dispatch("Excel.Application")
excel.Visible = True

workbook = excel.Workbooks.Open(excel_file)
workbook.RefreshAll()
excel.CalculateUntilAsyncQueriesDone()

win32.Dispatch("WScript.Shell").Popup("Proceso finalizado. Sigma Analytics", 5, "Aviso", 64)

c:\Users\carocha\OneDrive - Sigma\Archivos de Morales Mercado, Enrique Guadalupe - RL_Chedraui_y_ajustes_total\config.py:1: SyntaxWarning: invalid escape sequence '\S'
  server = 'SIACECLU04\SIACESQLQAS'
C:\Users\carocha\AppData\Local\Temp\ipykernel_24936\1619097351.py:29: DtypeWarning: Columns (0: Nº pedido cliente, 1: Cantidad.1, 2: Kg dentro MRP) have mixed types. Specify dtype option on import or set low_memory=False.
  df_transitos = pd.read_csv('transitos.csv', encoding='latin1')


Carpeta 'emergencia' ya existe.
Archivo eliminado: c:\Users\carocha\OneDrive - Sigma\Archivos de Morales Mercado, Enrique Guadalupe - RL_Chedraui_y_ajustes_total\emergencia\Ajustes_pedido.xlsx
Archivo eliminado: c:\Users\carocha\OneDrive - Sigma\Archivos de Morales Mercado, Enrique Guadalupe - RL_Chedraui_y_ajustes_total\emergencia\df_rol.csv
Archivo 'Ajustes_pedido.xlsx' copiado exitosamente a la carpeta 'emergencia'.
Archivo guardado en: c:\Users\carocha\OneDrive - Sigma\Archivos de Morales Mercado, Enrique Guadalupe - RL_Chedraui_y_ajustes_total\emergencia\df_rol.csv


-1

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
from conn_mysql import get_connection
import pyodbc
from config import server, database, username, password
from datetime import datetime
import locale
import time
from pathlib import Path

conn = get_connection()
query = "SELECT * FROM proyeccion_detalle_pedidos where id_cadena <>'123456'"
df_Oc_cliente = pd.read_sql(query, conn)
conn.close()


ModuleNotFoundError: No module named 'conn_mysql'

In [15]:
!pip install --upgrade --force-reinstall pywin32

Defaulting to user installation because normal site-packages is not writeable
  Using cached pywin32-311-cp313-cp313-win_amd64.whl.metadata (10 kB)
Using cached pywin32-311-cp313-cp313-win_amd64.whl (9.5 MB)
  Attempting uninstall: pywin32
    Found existing installation: pywin32 311
    Uninstalling pywin32-311:
      Successfully uninstalled pywin32-311



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\carocha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
